In [7]:
from bs4 import BeautifulSoup
import requests
import json
import re
import unicodedata


In [8]:

url = "https://fptshop.com.vn/dien-thoai"
html = requests.get(
    url,
    headers={
        "User-Agent": "Mozilla/5.0"
    }
).text

soup = BeautifulSoup(html, "html.parser")

In [9]:
def slugify(text):
    text = unicodedata.normalize("NFD", text)
    text = text.encode("ascii", "ignore").decode("utf-8")
    text = text.lower()

    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = re.sub(r"-+", "-", text)

    return text.strip("-")

In [13]:
brands = []

seen = set()

for img in soup.select("button img[alt]"):
    name = img.get("alt", "").strip()

    if not name:
        continue

    src = img.get("src")

    # chỉ lấy logo hãng
    if "logo_" not in src:
        continue

    if name in seen:
        continue

    seen.add(name)

    brands.append({
        "brand_name": name,
        "slug": slugify(name),
        "brand_logo_url": src
    })

print(json.dumps(brands, ensure_ascii=False, indent=2))
print(f"Đã tìm thấy {len(brands)} thương hiệu.")

[
  {
    "brand_name": "Apple",
    "slug": "apple",
    "brand_logo_url": "https://cdn2.fptshop.com.vn/unsafe/180x0/filters:format(webp):quality(75)/small/logo_apple_ngang_1810642801.png"
  },
  {
    "brand_name": "Samsung",
    "slug": "samsung",
    "brand_logo_url": "https://cdn2.fptshop.com.vn/unsafe/180x0/filters:format(webp):quality(75)/small/logo_samsung_ngang_1624d75bd8.png"
  },
  {
    "brand_name": "Xiaomi",
    "slug": "xiaomi",
    "brand_logo_url": "https://cdn2.fptshop.com.vn/unsafe/180x0/filters:format(webp):quality(75)/small/logo_xiaomi_ngang_0faf267234.png"
  },
  {
    "brand_name": "OPPO",
    "slug": "oppo",
    "brand_logo_url": "https://cdn2.fptshop.com.vn/unsafe/180x0/filters:format(webp):quality(75)/small/logo_oppo_ngang_68d31fcd73.png"
  },
  {
    "brand_name": "HONOR",
    "slug": "honor",
    "brand_logo_url": "https://cdn2.fptshop.com.vn/unsafe/180x0/filters:format(webp):quality(75)/small/logo_honor_ngang_814fca59e4.png"
  },
  {
    "brand_name": "Tecn

In [14]:
with open("brands.json", "w", encoding="utf-8") as f:
    json.dump(
        brands,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Đã lưu {len(brands)} thương hiệu vào brands.json")

Đã lưu 16 thương hiệu vào brands.json


In [104]:
def get_products(driver, n=None):

    products = []
    grows = driver.find_all("div.grow")
    print(f"Tìm thấy {len(grows)} grows trên trang.")

    cards = []
    for grow in grows:  # selector card sản phẩm
        cards = grow.select("div.group")  # selector card sản phẩm
        print(f"Tìm thấy {len(cards)} card sản phẩm trên trang.")
        
        if n is not None and len(cards) > n:
            cards = cards[:n]

        for card in cards:
            try:
                # ===== Tên sản phẩm =====
                name_tag = card.select_one("h3")
                if not name_tag:
                    continue

                product_name = name_tag.get_text(strip=True)

                # ===== URL + slug =====
                link_tag = card.select_one('a[href^="/dien-thoai/"]')
                if not link_tag:
                    continue

                product_url = link_tag["href"]
                slug = product_url.split("/")[-1]

                # ===== Brand =====
                brand_name = product_name.split()[0]

                # ===== Thumbnail =====
                img_tag = card.select_one('img[src*="fptshop"]')

                thumbnail_url = (
                    img_tag["src"]
                    if img_tag
                    else None
                )

                # ===== Giá gốc =====
                original_price = None

                original_price_tag = card.select_one(
                    ".line-through"
                )

                if original_price_tag:
                    original_price = int(
                        re.sub(r"\D", "", original_price_tag.text)
                    )

                # ===== Giá sale =====
                sale_price = None

                sale_price_tag = card.select_one(
                    "p.text-textOnWhitePrimary"
                )

                if sale_price_tag:
                    sale_price = int(
                        re.sub(r"\D", "", sale_price_tag.text)
                    )

                # ===== % giảm =====
                discount_percent = None

                discount_tag = card.select_one(
                    "span.text-textOnWhiteBrand"
                )

                if discount_tag:
                    discount_percent = int(
                        re.sub(r"\D", "", discount_tag.text)
                    )

                products.append({
                    "product_name": product_name,
                    "slug": slug,
                    "brand_name": brand_name,
                    "product_url": product_url,
                    "thumbnail_url": thumbnail_url,
                    "original_price": original_price,
                    "sale_price": sale_price,
                    "discount_percent": discount_percent
                })

            except Exception as e:
                print("Lỗi:", e)

    return products

In [108]:
import json
import re
import time
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By


def get_products(soup, n=None):
    """Hàm bóc tách dữ liệu sản phẩm đã cập nhật Selector chuẩn mới nhất của FPT

    Shop.
    """
    products = []

    # CẢI TIẾN 1: FPT Shop hiện tại quản lý card sản phẩm bằng thuộc tính data-testid hoặc class chứa 'product-item'
    # Chúng ta sẽ quét tất cả các thẻ <a> hoặc <div> là layout của Card sản phẩm.
    # Thử nghiệm selector phổ biến và bền vững nhất của FPT Shop:
    cards = soup.select('div[class*="product-item"]') or soup.select(
        "div.group"
    )

    # Nếu vẫn không tìm thấy, thử quét diện rộng qua cấu trúc thẻ chứa link chi tiết điện thoại
    if not cards:
        # Lấy các thẻ div cha bao quanh link điện thoại
        links = soup.select('a[href^="/dien-thoai/"]')
        # Loại bỏ các link trùng hoặc link không phải card (như banner, breadcrumb) bằng cách lấy thẻ div cha
        cards = [
            link.find_parent("div")
            for link in links
            if link.find_parent("div")
        ]
        # Gom cụm độc bản
        cards = list(set(cards))

    print(f"Hệ thống tìm thấy: {len(cards)} card sản phẩm tiềm năng.")

    for card in cards:
        if n is not None and len(products) >= n:
            break

        try:
            # ===== 1. URL + slug (Tìm thẻ a có chứa /dien-thoai/) =====
            # Đôi khi chính card là thẻ <a>, hoặc thẻ <a> nằm bên trong card
            if card.name == "a" and "/dien-thoai/" in card.get("href", ""):
                link_tag = card
            else:
                link_tag = card.select_one('a[href*="/dien-thoai/"]')

            if not link_tag or not link_tag.get("href"):
                continue

            product_url = link_tag["href"]
            # Chuẩn hóa nếu link là đường dẫn tương đối
            if product_url.startswith("/"):
                product_url = "https://fptshop.com.vn" + product_url

            slug = product_url.split("/")[-1]

            # ===== 2. Tên sản phẩm =====
            # FPT Shop thường để tên trong thẻ h3, hoặc thẻ div/p có thuộc tính text-ellipsis
            name_tag = (
                card.select_one("h3")
                or card.select_one('div[class*="title"]')
                or card.select_one('p[class*="name"]')
            )

            # Nếu không tìm thấy bằng class, lấy text hiển thị chính của thẻ link_tag
            if name_tag:
                raw_name = name_tag.get_text(strip=True)
            else:
                raw_name = link_tag.get_text(strip=True)

            if not raw_name or len(raw_name) < 3:
                continue

            # Thực hiện Regex làm sạch RAM, ROM ở cuối tên
            clean_name = re.sub(
                r"\s+\d+\s*(GB|TB).*$", "", raw_name, flags=re.IGNORECASE
            ).strip()

            # ===== 3. Thương hiệu =====
            brand_name = clean_name.split()[0] if clean_name else "Unknown"

            # ===== 4. Ảnh đại diện (Thumbnail) =====
            img_tag = card.select_one("img")
            thumbnail_url = None
            if img_tag:
                # Tránh lấy ảnh loading/lazyload (.svg), ưu tiên src hoặc data-src
                thumbnail_url = (
                    img_tag.get("data-src")
                    or img_tag.get("src")
                    or img_tag.get("lazy-src")
                )

            # Loại bỏ trường hợp cào nhầm icon SVG hình vuông nhỏ
            if thumbnail_url and ".svg" in thumbnail_url.lower():
                thumbnail_url = None

            # Đóng gói dữ liệu sạch
            products.append(
                {
                    "product_name": clean_name,
                    "slug": slug,
                    "brand_name": brand_name,
                    "product_url": product_url,
                    "thumbnail_url": thumbnail_url,
                }
            )

        except Exception as e:
            # Bỏ chú thích print lỗi này đi nếu log quá nhiều card rác không thỏa mãn
            pass

    return products


# =========================================================================
# LUỒNG CHẠY CHÍNH (MAIN PROCESS)
# =========================================================================

# 1. Khởi tạo trình duyệt Chrome tự động
driver = webdriver.Chrome()
driver.get("https://fptshop.com.vn/dien-thoai")
time.sleep(3)  # Đợi trang web tải xong

click_count = 0
last_page_source = ""

# 2. Vòng lặp bấm nút "Xem thêm" liên tục cho đến khi hết bài hoặc đủ tải
while True:
    try:
        button = driver.find_element(
            By.CSS_SELECTOR, "button.duration-300.rounded-3xl.px-4.py-2"
        )

        # Chống lặp vô hạn: Kiểm tra nếu cấu trúc trang không thay đổi sau khi bấm
        current_page_source = driver.page_source
        if current_page_source == last_page_source:
            print("--> HTML không đổi. Đã tải hết sản phẩm thực tế!")
            break
        last_page_source = current_page_source

        print("Đã tìm thấy nút 'Xem thêm', đang chuẩn bị bấm...")
        driver.execute_script("arguments[0].click();", button)

        click_count += 1
        print(f"Đã bấm nút lần thứ {click_count}...")

        time.sleep(3)  # Chờ 3 giây để dữ liệu mới render hoàn chỉnh

    except Exception as e:
        print("--> Đã bấm hết! Không còn tìm thấy nút 'Xem thêm' nữa.")
        break

# 3. CHỈ LẤY SOURCE KHI DRIVER CÒN SỐNG
print("Đang lấy mã nguồn toàn bộ trang web...")
full_page_html = driver.page_source

# 4. BÂY GIỜ MỚI ĐƯỢC PHÉP ĐÓNG DRIVER
driver.quit()
print("Đã đóng trình duyệt an toàn.")

# 5. Dùng BeautifulSoup để cào dữ liệu từ biến HTML tĩnh đã lưu
soup = BeautifulSoup(full_page_html, "html.parser")
products = get_products(soup, n=200)

print("\n[KẾT QUẢ DANH SÁCH SẢN PHẨM HOÀN CHỈNH]")
print(json.dumps(products, ensure_ascii=False, indent=2))
print(f"\nTổng kết: Đã tìm thấy {len(products)} sản phẩm.")

Đã tìm thấy nút 'Xem thêm', đang chuẩn bị bấm...
Đã bấm nút lần thứ 1...
Đã tìm thấy nút 'Xem thêm', đang chuẩn bị bấm...
Đã bấm nút lần thứ 2...
Đã tìm thấy nút 'Xem thêm', đang chuẩn bị bấm...
Đã bấm nút lần thứ 3...
Đã tìm thấy nút 'Xem thêm', đang chuẩn bị bấm...
Đã bấm nút lần thứ 4...
Đã tìm thấy nút 'Xem thêm', đang chuẩn bị bấm...
Đã bấm nút lần thứ 5...
Đã tìm thấy nút 'Xem thêm', đang chuẩn bị bấm...
Đã bấm nút lần thứ 6...
Đã tìm thấy nút 'Xem thêm', đang chuẩn bị bấm...
Đã bấm nút lần thứ 7...
--> Đã bấm hết! Không còn tìm thấy nút 'Xem thêm' nữa.
Đang lấy mã nguồn toàn bộ trang web...
Đã đóng trình duyệt an toàn.
Hệ thống tìm thấy: 168 card sản phẩm tiềm năng.

[KẾT QUẢ DANH SÁCH SẢN PHẨM HOÀN CHỈNH]
[
  {
    "product_name": "Nubia A76",
    "slug": "nubia-a76",
    "brand_name": "Nubia",
    "product_url": "https://fptshop.com.vn/dien-thoai/nubia-a76",
    "thumbnail_url": "https://cdn2.fptshop.com.vn/unsafe/360x0/filters:format(webp):quality(75)/nubia_a76_xam_5_87aade2a

In [47]:
with open("products.json", "w", encoding="utf-8") as f:
    json.dump(
        products,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Đã lưu {len(products)} thương hiệu vào products.json")

Đã lưu 154 thương hiệu vào products.json


In [91]:
product = products[2] if products else None
print(json.dumps(product, ensure_ascii=False, indent=2))

{
  "product_name": "Samsung Galaxy S26 Ultra 5G 12GB 256GB",
  "slug": "samsung-galaxy-s26-ultra",
  "brand_name": "Samsung",
  "product_url": "/dien-thoai/samsung-galaxy-s26-ultra",
  "thumbnail_url": "https://cdn2.fptshop.com.vn/unsafe/360x0/filters:format(webp):quality(75)/samsung_galaxy_s26_ultra_tim_d3898ec641.png",
  "original_price": 36990000,
  "sale_price": 30190000,
  "discount_percent": 18
}


In [92]:
url_product = "https://fptshop.com.vn" + product["product_url"]
print(url_product)

https://fptshop.com.vn/dien-thoai/samsung-galaxy-s26-ultra


In [ ]:
import json
import time
from selenium import webdriver
from selenium.webdriver.common.by import By

url_product = "https://fptshop.com.vn" + product["product_url"]
print(url_product)

driver = webdriver.Chrome()
driver.get(url_product)
time.sleep(3)

try:
    # =========================================================================
    # LẤY THÔNG TIN BREADCRUMB & SERIES
    # =========================================================================
    breadcrumb = driver.find_element(By.CSS_SELECTOR, "nav.Breadcrumb")
    series = breadcrumb.text.split("\n")[-1] if "\n" in breadcrumb.text else None
    print("Series:", series)

    # =========================================================================
    # XỬ LÝ LỌC THEO DUNG LƯỢNG VÀ LẤY GIÁ BÁN TƯƠNG ỨNG
    # =========================================================================
    storage_buttons = driver.find_elements(
        By.XPATH,
        "//div[contains(@class, 'flex')][span[text()='Dung lượng']]//button"
    )

    storage_and_price_results = []

    for idx, storage_btn in enumerate(storage_buttons):
        try:
            storage_name = storage_btn.find_element(By.XPATH, ".//span[1]").text.strip()
        except:
            continue

        print(f"\n===== Đang kích hoạt cấu hình Dung lượng: {storage_name} =====")
        driver.execute_script("arguments[0].click();", storage_btn)
        time.sleep(2)  # Đợi hệ thống render lại toàn bộ giá tiền, màu sắc và RAM

        # ĐÃ THÊM: Bóc tách dung lượng RAM tương ứng của phiên bản bộ nhớ này
        try:
            ram_tag = driver.find_element(
                By.XPATH, 
                "//div[p[text()='RAM']]//p[contains(@class, 'b1-semibold')]"
            )
            ram_value = ram_tag.text.strip()
        except:
            ram_value = "Không rõ" # Phòng trường hợp máy tính bảng/máy không hiển thị nhãn RAM bên ngoài

        # Bóc tách cụm thông tin giá bán
        try:
            current_price = driver.find_element(By.XPATH, "//span[contains(@class, 'h4-bold')]").text.strip()
        except:
            current_price = "Liên hệ"

        try:
            original_price = driver.find_element(By.XPATH, "//span[contains(@class, 'line-through')]").text.strip()
        except:
            original_price = "None"

        try:
            discount_percentage = driver.find_element(By.XPATH, "//span[contains(@class, 'text-red-red-7')]").text.strip()
        except:
            discount_percentage = "0%"

        # ---------------------------------------------------------------------
        # XỬ LÝ LẤY MÀU SẮC & ẢNH LỚN CHO RIÊNG PHIÊN BẢN DUNG LƯỢNG NÀY
        # ---------------------------------------------------------------------
        color_section = driver.find_element(By.XPATH, "//div[contains(@class, 'flex')][span[text()='Màu sắc']]")
        color_buttons = color_section.find_elements(By.XPATH, ".//button")

        color_variants = []
        for color_btn in color_buttons:
            btn_text = color_btn.text.strip()
            if not btn_text: continue
            color_name = btn_text.split("\n")[0]

            driver.execute_script("arguments[0].click();", color_btn)
            time.sleep(1.5)

            try:
                avatar_img = color_btn.find_element(By.XPATH, ".//img")
                avatar_url = avatar_img.get_attribute("src")
                color_image_url = avatar_url.replace("/64x0/", "/750x0/").replace("/32x0/", "/750x0/")
            except:
                color_image_url = "No Avatar"

            thumb_images = driver.find_elements(
                By.XPATH, "//div[contains(@class, 'thumb-container')]//div[contains(@class, 'swiper-slide')]//img"
            )

            image_list = []
            for img in thumb_images:
                src = img.get_attribute("src")
                if src:
                    if ".svg" in src.lower(): continue
                    if "cdn2.fptshop.com.vn" in src or "fptshop.com.vn" in src:
                        large_src = src.replace("/128x0/", "/1920x0/").replace("/64x0/", "/1920x0/")
                        if large_src not in image_list:
                            image_list.append(large_src)

            color_variants.append({
                "color": color_name,
                "color_image_url": color_image_url,
                "images": image_list
            })

        # Đóng gói cấu hình dữ liệu (Đã bổ sung thuộc tính RAM vào object)
        storage_and_price_results.append({
            "storage": storage_name,
            "ram": ram_value,  # <--- Ghi nhận RAM tại đây
            "current_price": current_price,
            "original_price": original_price,
            "discount_percentage": discount_percentage,
            "color_variants": color_variants
        })

    # In kết quả cấu trúc Sản phẩm biến thể kèm giá và RAM
    print("\n[KẾT QUẢ DỮ LIỆU BIẾN THỂ VÀ GIÁ BÁN]")
    print(json.dumps(storage_and_price_results, ensure_ascii=False, indent=2))

    # =========================================================================
    # THÔNG SỐ KỸ THUẬT (Quét 1 lần duy nhất ở cuối)
    # =========================================================================
    print("\nĐang lấy thông số kỹ thuật toàn bộ...")
    spec_button = driver.find_element(By.XPATH, "//button[contains(., 'Xem tất cả thông số')]")
    driver.execute_script("arguments[0].click();", spec_button)
    time.sleep(2)

    spec_blocks = driver.find_elements(
        By.XPATH, "//div[contains(@class, 'tab-content') and starts-with(@id, 'spec-item-')]"
    )

    specs_json = []
    for block in spec_blocks:
        try:
            category_name = block.find_element(By.XPATH, "./div[contains(@class, 'b2-semibold')]").text.strip()
        except:
            continue

        category_data = {"category": category_name, "specifications": {}}
        rows = block.find_elements(By.XPATH, "./div[contains(@class, 'border-dashed')]")
        
        for row in rows:
            try:
                key = row.find_element(By.XPATH, "./div[contains(@class, 'w-2/5')]").text.strip()
                value_text = row.find_element(By.XPATH, "./*[2]").text.strip().replace("\n", ", ")
                if key and value_text:
                    category_data["specifications"][key] = value_text
            except:
                continue

        if category_data["specifications"]:
            specs_json.append(category_data)

    print("\n[KẾT QUẢ CÀO THÔNG SỐ KỸ THUẬT DẠNG JSON]")
    print(json.dumps(specs_json, ensure_ascii=False, indent=2))

    # =========================================================================
    # LƯU MÔ TẢ HTML SẢN PHẨM
    # =========================================================================
    description_element = driver.find_element(By.ID, "MoTaSanPham")
    description_html = description_element.get_attribute("innerHTML").replace("/800x0/", "/1920x0/")
    print("\n--- Đã lấy thành công cấu trúc HTML Mô tả ---")
    print(description_html[:500] + "...")

except Exception as e:
    print("Lỗi khi truy cập trang sản phẩm:", e)

driver.quit()

Series: Galaxy S Series

===== Đang kích hoạt cấu hình Dung lượng: 256 GB =====

===== Đang kích hoạt cấu hình Dung lượng: 512 GB =====

[KẾT QUẢ DỮ LIỆU BIẾN THỂ VÀ GIÁ BÁN]
[
  {
    "storage": "256 GB",
    "ram": "12 GB",
    "current_price": "30.190.000đ",
    "original_price": "36.990.000đ",
    "discount_percentage": "18%",
    "color_variants": [
      {
        "color": "Đen",
        "color_image_url": "https://cdn2.fptshop.com.vn/unsafe/750x0/filters:format(webp):quality(75)/samsung_galaxy_s26_ultra_den_cd0249a59b.png",
        "images": [
          "https://cdn2.fptshop.com.vn/unsafe/1920x0/filters:format(webp):quality(75)/samsung_galaxy_s26_ultra_den_cd0249a59b.png",
          "https://cdn2.fptshop.com.vn/unsafe/1920x0/filters:format(webp):quality(75)/samsung_galaxy_s26_ultra_black_2_313c53ed1f.png",
          "https://cdn2.fptshop.com.vn/unsafe/1920x0/filters:format(webp):quality(75)/samsung_galaxy_s26_ultra_black_3_c6f362207e.png",
          "https://cdn2.fptshop.com.vn/

In [ ]:
import json
import re

with open("products.json", "r", encoding="utf-8") as f:
    products = json.load(f)

for product in products:
    # Xóa các trường giá
    product.pop("original_price", None)
    product.pop("sale_price", None)
    product.pop("discount_percent", None)

    # Chuẩn hóa tên sản phẩm
    name = product["product_name"]

    # Loại bỏ phần RAM/Dung lượng ở cuối tên
    name = re.sub(
        r"\s+\d+GB(\s+\d+GB)*$",
        "",
        name,
        flags=re.IGNORECASE
    )

    product["product_name"] = name.strip()

with open("products_clean.json", "w", encoding="utf-8") as f:
    json.dump(products, f, ensure_ascii=False, indent=2)

print("Đã xử lý xong!")
print()

Đã xử lý xong!


In [111]:
import json
import time
from selenium.webdriver.common.by import By


def crawl_product_detail(driver, product):
    url_product = "https://fptshop.com.vn" + product["product_url"]

    result = {
        **product,
        "series": None,
        "variants": [],
        "specifications": [],
        "description_html": None
    }

    try:
        driver.get(url_product)
        time.sleep(3)

        # ==========================================================
        # BREADCRUMB / SERIES
        # ==========================================================
        try:
            breadcrumb = driver.find_element(
                By.CSS_SELECTOR,
                "nav.Breadcrumb"
            )

            result["series"] = (
                breadcrumb.text.split("\n")[-1]
                if "\n" in breadcrumb.text
                else None
            )
        except:
            pass

        # ==========================================================
        # BIẾN THỂ DUNG LƯỢNG / RAM / MÀU / GIÁ
        # ==========================================================
        storage_buttons = driver.find_elements(
            By.XPATH,
            "//div[contains(@class,'flex')][span[text()='Dung lượng']]//button"
        )

        variants = []

        for storage_btn in storage_buttons:

            try:
                storage_name = storage_btn.find_element(
                    By.XPATH,
                    ".//span[1]"
                ).text.strip()

            except:
                continue

            driver.execute_script(
                "arguments[0].click();",
                storage_btn
            )

            time.sleep(2)

            # RAM
            try:
                ram_value = driver.find_element(
                    By.XPATH,
                    "//div[p[text()='RAM']]//p[contains(@class,'b1-semibold')]"
                ).text.strip()
            except:
                ram_value = None

            # Giá hiện tại
            try:
                current_price = driver.find_element(
                    By.XPATH,
                    "//span[contains(@class,'h4-bold')]"
                ).text.strip()
            except:
                current_price = None

            # Giá gốc
            try:
                original_price = driver.find_element(
                    By.XPATH,
                    "//span[contains(@class,'line-through')]"
                ).text.strip()
            except:
                original_price = None

            # % giảm giá
            try:
                discount_percentage = driver.find_element(
                    By.XPATH,
                    "//span[contains(@class,'text-red-red-7')]"
                ).text.strip()
            except:
                discount_percentage = None

            # ======================================================
            # MÀU SẮC
            # ======================================================
            color_variants = []

            try:
                color_section = driver.find_element(
                    By.XPATH,
                    "//div[contains(@class,'flex')][span[text()='Màu sắc']]"
                )

                color_buttons = color_section.find_elements(
                    By.XPATH,
                    ".//button"
                )

                for color_btn in color_buttons:

                    color_text = color_btn.text.strip()

                    if not color_text:
                        continue

                    color_name = color_text.split("\n")[0]

                    driver.execute_script(
                        "arguments[0].click();",
                        color_btn
                    )

                    time.sleep(1.5)

                    try:
                        avatar_img = color_btn.find_element(
                            By.XPATH,
                            ".//img"
                        )

                        avatar_url = avatar_img.get_attribute("src")

                        color_image_url = (
                            avatar_url
                            .replace("/64x0/", "/750x0/")
                            .replace("/32x0/", "/750x0/")
                        )

                    except:
                        color_image_url = None

                    thumb_images = driver.find_elements(
                        By.XPATH,
                        "//div[contains(@class,'thumb-container')]//img"
                    )

                    image_list = []

                    for img in thumb_images:

                        src = img.get_attribute("src")

                        if not src:
                            continue

                        if ".svg" in src.lower():
                            continue

                        large_src = (
                            src
                            .replace("/128x0/", "/1920x0/")
                            .replace("/64x0/", "/1920x0/")
                        )

                        if large_src not in image_list:
                            image_list.append(large_src)

                    color_variants.append({
                        "color": color_name,
                        "color_image_url": color_image_url,
                        "images": image_list
                    })

            except:
                pass

            variants.append({
                "storage": storage_name,
                "ram": ram_value,
                "current_price": current_price,
                "original_price": original_price,
                "discount_percentage": discount_percentage,
                "color_variants": color_variants
            })

        result["variants"] = variants

        # ==========================================================
        # THÔNG SỐ KỸ THUẬT
        # ==========================================================
        try:
            spec_button = driver.find_element(
                By.XPATH,
                "//button[contains(., 'Xem tất cả thông số')]"
            )

            driver.execute_script(
                "arguments[0].click();",
                spec_button
            )

            time.sleep(2)

            spec_blocks = driver.find_elements(
                By.XPATH,
                "//div[contains(@class,'tab-content') and starts-with(@id,'spec-item-')]"
            )

            specs_json = []

            for block in spec_blocks:

                try:
                    category_name = block.find_element(
                        By.XPATH,
                        "./div[contains(@class,'b2-semibold')]"
                    ).text.strip()

                except:
                    continue

                category = {
                    "category": category_name,
                    "specifications": {}
                }

                rows = block.find_elements(
                    By.XPATH,
                    "./div[contains(@class,'border-dashed')]"
                )

                for row in rows:

                    try:
                        key = row.find_element(
                            By.XPATH,
                            "./div[contains(@class,'w-2/5')]"
                        ).text.strip()

                        value = row.find_element(
                            By.XPATH,
                            "./*[2]"
                        ).text.strip().replace("\n", ", ")

                        category["specifications"][key] = value

                    except:
                        continue

                specs_json.append(category)

            result["specifications"] = specs_json

        except:
            pass

        # ==========================================================
        # HTML MÔ TẢ
        # ==========================================================
        try:
            description_element = driver.find_element(
                By.ID,
                "MoTaSanPham"
            )

            result["description_html"] = (
                description_element
                .get_attribute("innerHTML")
                .replace("/800x0/", "/1920x0/")
            )

        except:
            pass

        return result

    except Exception as e:

        print(
            f"Lỗi khi crawl {url_product}:",
            e
        )

        return None

In [114]:
import json
import re
from selenium import webdriver


# ==============================
# HELPER FUNCTIONS
# ==============================

def parse_price(text):
    if not text:
        return None

    digits = re.sub(r"\D", "", str(text))

    return int(digits) if digits else None


def parse_ram(text):
    if not text:
        return None

    match = re.search(r"(\d+)\s*GB", text, re.IGNORECASE)

    return int(match.group(1)) if match else None


def parse_storage(text):
    if not text:
        return None

    match = re.search(r"(\d+)\s*GB", text, re.IGNORECASE)

    return int(match.group(1)) if match else None


def parse_discount(text):
    if not text:
        return None

    match = re.search(r"(\d+)", str(text))

    return int(match.group(1)) if match else None


# ==============================
# TRANSFORM DATA
# ==============================

def transform_product(detail):
    """
    Chuyển dữ liệu từ crawl_product_detail()
    sang schema đích
    """

    product_json = {
        "brand_name": detail.get("brand_name"),
        "series_name": detail.get("series"),
        "category_name": "Điện thoại",

        "product_info": {
            "name": detail.get("product_name"),
            "base_name": detail.get("product_name"),
            "short_description": None,
            "detail_description": detail.get("description_html"),
            "thumbnail_url": detail.get("thumbnail_url"),
            "sale": None,
            "warranty_months": 12,
            "status": "ACTIVE",
            "is_featured": False
        },

        "variants": [],
        "specifications": []
    }

    # ==========================
    # VARIANTS
    # ==========================

    for variant in detail.get("variants", []):

        ram_gb = parse_ram(
            variant.get("ram")
        )

        storage_gb = parse_storage(
            variant.get("storage")
        )

        price = parse_price(
            variant.get("current_price")
        )

        compare_price = parse_price(
            variant.get("original_price")
        )

        discount_percent = parse_discount(
            variant.get("discount_percentage")
        )

        for color_variant in variant.get(
            "color_variants",
            []
        ):

            images = []

            for index, image_url in enumerate(
                color_variant.get("images", []),
                start=1
            ):

                images.append({
                    "image_url": image_url,
                    "is_primary": index == 1,
                    "sort_order": index
                })

            product_json["variants"].append({
                "sku": None,

                "color":
                    color_variant.get("color"),

                "ram_gb":
                    ram_gb,

                "storage_label":
                    variant.get("storage"),

                "storage_gb":
                    storage_gb,

                "price":
                    price,

                "compare_at_price":
                    compare_price,

                "cost_price":
                    None,

                "weight_gram":
                    None,

                "barcode":
                    None,

                "color_image_url":
                    color_variant.get(
                        "color_image_url"
                    ),

                "images":
                    images,

                "discount":
                    {
                        "discount_percent":
                            discount_percent,

                        "discount_amount":
                            None,

                        "discount_type":
                            "PERCENT",

                        "start_at":
                            None,

                        "end_at":
                            None
                    }
                    if discount_percent
                    else None
            })

    # ==========================
    # SPECIFICATIONS
    # ==========================

    sort_order = 1

    for category in detail.get(
        "specifications",
        []
    ):

        category_name = category.get(
            "category"
        )

        specs = category.get(
            "specifications",
            {}
        )

        for key, value in specs.items():

            product_json[
                "specifications"
            ].append({
                "spec_category":
                    category_name,

                "spec_key":
                    key,

                "spec_value":
                    value,

                "sort_order":
                    sort_order
            })

            sort_order += 1

    return product_json


# ==============================
# MAIN
# ==============================

def main():

    # Đọc danh sách sản phẩm
    with open(
        "products.json",
        "r",
        encoding="utf-8"
    ) as f:

        products = json.load(f)

    driver = webdriver.Chrome()

    final_products = []

    try:

        for index, product in enumerate(
            products,
            start=1
        ):

            print(
                f"[{index}/{len(products)}] "
                f"Đang xử lý: "
                f"{product['product_name']}"
            )

            try:

                detail = crawl_product_detail(
                    driver,
                    product
                )

                if not detail:
                    continue

                transformed = transform_product(
                    detail
                )

                final_products.append(
                    transformed
                )

            except Exception as e:

                print(
                    "Lỗi:",
                    product["product_name"],
                    e
                )

        # Lưu file kết quả

        with open(
            "products_final.json",
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                final_products,
                f,
                ensure_ascii=False,
                indent=2
            )

        print(
            f"\nHoàn thành "
            f"{len(final_products)} sản phẩm"
        )

    finally:

        driver.quit()


# ==============================
# RUN
# ==============================

if __name__ == "__main__":
    main()

[1/154] Đang xử lý: Xiaomi 17T 5G
[2/154] Đang xử lý: Xiaomi 17T Pro 5G
[3/154] Đang xử lý: Samsung Galaxy S26 Ultra 5G
[4/154] Đang xử lý: Oppo Find X9 Ultra 5G
[5/154] Đang xử lý: Oppo Find X9s 5G
[6/154] Đang xử lý: iPhone 17 Pro Max
[7/154] Đang xử lý: Samsung Galaxy Z Fold7 5G
[8/154] Đang xử lý: OPPO Reno15 F 5G
[9/154] Đang xử lý: Xiaomi Redmi Note 15
[10/154] Đang xử lý: iPhone 17
[11/154] Đang xử lý: Honor X9d 5G
[12/154] Đang xử lý: Samsung Galaxy S25 Ultra 5G
[13/154] Đang xử lý: REDMAGIC 11 Pro 5G
[14/154] Đang xử lý: Xiaomi Poco X7 5G
[15/154] Đang xử lý: Samsung Galaxy A17
[16/154] Đang xử lý: Honor X7d
[17/154] Đang xử lý: iPhone 17 Pro
[18/154] Đang xử lý: Xiaomi Redmi 13x
[19/154] Đang xử lý: Samsung Galaxy A07
[20/154] Đang xử lý: Samsung Galaxy S25 FE 5G
[21/154] Đang xử lý: Honor X6c
[22/154] Đang xử lý: Nubia A76 4GB 128GB (NFC)
[23/154] Đang xử lý: Xiaomi Poco M7 Pro 5G
[24/154] Đang xử lý: Nubia V70 Design
[25/154] Đang xử lý: Masstel Izi 15 4G
[26/154] Đang xử l

In [118]:
import json
import cloudinary
import cloudinary.uploader
from bs4 import BeautifulSoup

# 1. Cấu hình Cloudinary
cloudinary.config(
    cloud_name="dozpywcus",
    api_key="345466748819321",
    api_secret="hX6w0WmgPjD1ffxaDXs69QUjiDQ"
)

def upload_to_cloudinary(image_url):
    """
    Hàm thực hiện upload một URL ảnh từ xa lên Cloudinary và trả về URL mới.
    Nếu có lỗi hoặc url trống, hoặc ảnh đã thuộc cloudinary thì trả lại url gốc.
    """
    if not image_url or "cloudinary.com" in image_url:
        return image_url
    try:
        # Upload trực tiếp bằng URL từ xa
        response = cloudinary.uploader.upload(
            image_url,
            folder="products"  # Gom ảnh vào thư mục 'products' trên Cloudinary
        )
        print(f"  [Success]: {image_url} ---> {response['secure_url']}")
        return response['secure_url']
    except Exception as e:
        print(f"  [Error] khi upload {image_url}: {e}")
        return image_url


def process_html_description(html_content):
    """
    Hàm bóc tách chuỗi HTML, tìm tất cả thẻ <img> để upload lên Cloudinary và thay thế link
    """
    if not html_content:
        return html_content
    
    soup = BeautifulSoup(html_content, "html.parser")
    images_in_html = soup.find_all("img")
    
    if not images_in_html:
        return html_content
        
    print(f"\n--- Đang xử lý ảnh trong mô tả chi tiết ({len(images_in_html)} ảnh) ---")
    for img in images_in_html:
        old_src = img.get("src")
        if old_src:
            new_src = upload_to_cloudinary(old_src)
            img["src"] = new_src
            
    return str(soup)


# === QUY TRÌNH ĐỌC FILE VÀ CHUYỂN ĐỔI ẢNH ===

input_file = "products_final.json"
output_file = "products_cloudinary.json"

try:
    # Đọc dữ liệu từ file JSON gốc
    print(f"--- Đang đọc dữ liệu từ file '{input_file}' ---")
    with open(input_file, "r", encoding="utf-8") as f:
        products = json.load(f)
    print(f"Đã tải thành công {len(products)} sản phẩm từ file.\n")
    
    print("=== BẮT ĐẦU QUY TRÌNH CHUYỂN ĐỔI ẢNH LÊN CLOUDINARY ===")

    for index, product in enumerate(products):
        brand = product.get("brand_name", "Unknown")
        prod_info = product.get("product_info", {})
        p_name = prod_info.get("name", f"Product #{index}")
        
        print(f"\n>> Đang xử lý [{index + 1}/{len(products)}]: [{brand}] {p_name}")
        
        # Xử lý thumbnail_url của sản phẩm
        if "thumbnail_url" in prod_info and prod_info["thumbnail_url"]:
            prod_info["thumbnail_url"] = upload_to_cloudinary(prod_info["thumbnail_url"])
            
        # Xử lý detail_description (các ảnh nằm trong mã HTML)
        if "detail_description" in prod_info and prod_info["detail_description"]:
            prod_info["detail_description"] = process_html_description(prod_info["detail_description"])
            
        # Xử lý danh sách các variants
        variants = product.get("variants", [])
        for v_idx, variant in enumerate(variants):
            color = variant.get("color", f"Variant #{v_idx}")
            print(f"  - Phiên bản màu: {color}")
            
            # Xử lý color_image_url
            if "color_image_url" in variant and variant["color_image_url"]:
                variant["color_image_url"] = upload_to_cloudinary(variant["color_image_url"])
                
            # Xử lý mảng ảnh chi tiết (images) bên trong mỗi variant
            variant_images = variant.get("images", [])
            for img_obj in variant_images:
                if "image_url" in img_obj and img_obj["image_url"]:
                    img_obj["image_url"] = upload_to_cloudinary(img_obj["image_url"])

    print("\n=== QUY TRÌNH HOÀN THÀNH ===")

    # Xuất kết quả sau khi chuyển đổi thành công ra file mới
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(products, f, ensure_ascii=False, indent=2)
    print(f"\nĐã xuất toàn bộ dữ liệu mới ra file '{output_file}' thành công!")

except FileNotFoundError:
    print(f"Lỗi: Không tìm thấy file '{input_file}' ở cùng thư mục chạy script này.")
except json.JSONDecodeError:
    print(f"Lỗi: File '{input_file}' không đúng định dạng cấu trúc JSON.")
except Exception as e:
    print(f"Lỗi hệ thống trong quá trình chạy: {e}")

--- Đang đọc dữ liệu từ file 'products_final.json' ---
Đã tải thành công 154 sản phẩm từ file.

=== BẮT ĐẦU QUY TRÌNH CHUYỂN ĐỔI ẢNH LÊN CLOUDINARY ===

>> Đang xử lý [1/154]: [Xiaomi] Xiaomi 17T 5G
  [Success]: https://cdn2.fptshop.com.vn/unsafe/360x0/filters:format(webp):quality(75)/xiaomi_17t_tim_5_f5b6890a7c.png ---> https://res.cloudinary.com/dozpywcus/image/upload/v1780202536/products/nq5hl22jdwvihkazhes5.webp
  - Phiên bản màu: Đen
  [Success]: https://cdn2.fptshop.com.vn/unsafe/1920x0/filters:format(webp):quality(75)/xiaomi_17t_den_5_22efd7dc8d.png ---> https://res.cloudinary.com/dozpywcus/image/upload/v1780202537/products/kumjrgyrju9e3zkk2upu.webp
  [Success]: https://cdn2.fptshop.com.vn/unsafe/1920x0/filters:format(webp):quality(75)/xiaomi_17t_den_1_d279a9d2c9.png ---> https://res.cloudinary.com/dozpywcus/image/upload/v1780202539/products/agze9bhi0bthqltnanou.webp
  [Success]: https://cdn2.fptshop.com.vn/unsafe/1920x0/filters:format(webp):quality(75)/xiaomi_17t_den_2_c4117e81

In [119]:
import json


def merge_specifications(final_path, cloudinary_path, output_path):
    # 1. Đọc dữ liệu từ file chứa thông số chuẩn
    with open(final_path, "r", encoding="utf-8") as f:
        final_data = json.load(f)

    # 2. Đọc dữ liệu từ file chứa ảnh Cloudinary (đang bị trống specs)
    with open(cloudinary_path, "r", encoding="utf-8") as f:
        cloudinary_data = json.load(f)

    # Đảm bảo dữ liệu luôn ở dạng list để dễ duyệt
    if isinstance(final_data, dict):
        final_data = [final_data]
    if isinstance(cloudinary_data, dict):
        cloudinary_data = [cloudinary_data]

    matched_count = 0

    # 3. Duyệt qua từng sản phẩm trong file Cloudinary để cập nhật
    for c_prod in cloudinary_data:
        c_name = c_prod.get("product_info", {}).get("name", "").strip()
        if not c_name:
            continue

        # Tìm sản phẩm tương ứng trong file products_final
        found_specs = None
        for f_prod in final_data:
            f_name = f_prod.get("product_info", {}).get("name", "").strip()

            # Kiểm tra nếu tên khớp nhau hoàn toàn hoặc tên file này chứa tên file kia
            if c_name in f_name or f_name in c_name:
                found_specs = f_prod.get("specifications", [])
                break

        # Nếu tìm thấy thông số phù hợp, tiến hành ghi đè thay thế mảng rỗng
        if found_specs is not None:
            c_prod["specifications"] = found_specs
            matched_count += 1
        else:
            print(
                f"⚠️ Không tìm thấy thông số phù hợp cho sản phẩm: {c_name}"
            )

    # 4. Ghi dữ liệu mới đã gộp vào file kết quả
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(cloudinary_data, f, ensure_ascii=False, indent=2)

    print(
        f"\n=== HOÀN THÀNH ==="
    )
    print(
        f"Đã cập nhật thông số thành công cho {matched_count}/{len(cloudinary_data)} sản phẩm."
    )
    print(f"Kết quả lưu tại file: '{output_path}'")


# Cấu hình đường dẫn file của bạn tại đây
if __name__ == "__main__":
    FINAL_FILE = "products_final.json"
    CLOUDINARY_FILE = "products_cloudinary.json"
    OUTPUT_FILE = "products_merged_output.json"  # Tạo file mới để tránh ghi đè lỗi vào file gốc

    merge_specifications(FINAL_FILE, CLOUDINARY_FILE, OUTPUT_FILE)


=== HOÀN THÀNH ===
Đã cập nhật thông số thành công cho 154/154 sản phẩm.
Kết quả lưu tại file: 'products_merged_output.json'


In [116]:
!pip install cloudinary